### TASK 1

### 1.2 Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10
})

RANDOM_STATE = 42
print('Libraries loaded successfully.')

Libraries loaded successfully.


### 1.3 Load Dataset

In [4]:
df = pd.read_excel('Online Retail.xlsx', engine='openpyxl')
print(f'Raw dataset shape: {df.shape}')
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
df.head()

Raw dataset shape: (541909, 8)
Column names: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate    float64
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,40513.351389,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,40513.351389,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,40513.351389,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,40513.351389,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,40513.351389,3.39,17850.0,United Kingdom


### 1.4 Data Pre-processing

In [5]:
# Step 1: Check missing values and drop missing CustomerID
print('Missing values before cleaning:')
print(df.isnull().sum())
df_clean = df.dropna(subset=['CustomerID']).copy()
print(f'\nRows after dropping missing CustomerID: {len(df_clean):,}')

Missing values before cleaning:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Rows after dropping missing CustomerID: 406,829


In [6]:
# Step 2: Remove cancellations (InvoiceNo starts with C)
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Rows after removing cancellations: {len(df_clean):,}')

# Step 3: Remove non-positive Quantity and UnitPrice
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f'Rows after removing invalid Quantity/UnitPrice: {len(df_clean):,}')

# Step 4: Parse InvoiceDate
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Step 5: Derive TotalPrice
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# CustomerID as integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print('\nCleaned dataset sample:')
df_clean.head(3)

Rows after removing cancellations: 397,924
Rows after removing invalid Quantity/UnitPrice: 397,884

Cleaned dataset sample:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,1970-01-01 00:00:00.000040513,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,1970-01-01 00:00:00.000040513,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,1970-01-01 00:00:00.000040513,2.75,17850,United Kingdom,22.00


In [7]:
# Step 6: Build RFM table
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date (reference for Recency): {snapshot_date.date()}')

rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

print(f'RFM table shape: {rfm.shape}')
print('\nRFM summary statistics:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))

Snapshot date (reference for Recency): 1970-01-02
RFM table shape: (4338, 4)

RFM summary statistics:
       Recency  Frequency   Monetary
count   4338.0    4338.00    4338.00
mean       1.0       4.27    2054.27
std        0.0       7.70    8989.23
min        1.0       1.00       3.75
25%        1.0       1.00     307.41
50%        1.0       2.00     674.48
75%        1.0       5.00    1661.74
max        1.0     209.00  280206.02


In [8]:
# Step 7: Winsorise outliers at 99th percentile
for col in ['Recency', 'Frequency', 'Monetary']:
    cap = rfm[col].quantile(0.99)
    rfm[col] = rfm[col].clip(upper=cap)
    print(f'{col} capped at 99th percentile: {cap:.2f}')

# Step 8: Standardise
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print('\nStandardisation complete.')
print(f'Mean per feature (should be ~0): {rfm_scaled.mean(axis=0).round(4)}')
print(f'Std per feature  (should be ~1): {rfm_scaled.std(axis=0).round(4)}')

Recency capped at 99th percentile: 1.00
Frequency capped at 99th percentile: 30.00
Monetary capped at 99th percentile: 19881.00

Standardisation complete.
Mean per feature (should be ~0): [ 0. -0.  0.]
Std per feature  (should be ~1): [0. 1. 1.]
